# 01 - Chat models and messages

Welcome to notebook one. The goal here is small but important: get a feel for the *shape* of LangChain. Once you see how it wraps any chat model behind one interface and represents conversations as a list of typed messages, the rest of the library is mostly variations on that idea.

By the end of this notebook you should be able to:

- swap providers (OpenAI, Anthropic, Google) by changing a single string
- read what an `AIMessage` actually contains and why each field matters
- pick the right method for the job: `invoke`, `stream`, or `batch`

Built against the official docs at [docs.langchain.com/oss/python/langchain](https://docs.langchain.com/oss/python/langchain/overview).

## Setup

If you have not installed the requirements yet, uncomment the pip line in the next cell. We load environment variables from `.env` so we never have to think about API keys again for the rest of the notebook.

In [ ]:
# If you have not installed yet, uncomment one of these.
# Recommended: uv (fast, modern). https://docs.astral.sh/uv/
# !uv pip install -q langchain langchain-openai langchain-anthropic python-dotenv
# Or with plain pip:
# !pip install -q langchain langchain-openai langchain-anthropic python-dotenv

from dotenv import load_dotenv
load_dotenv()

import os
print("OpenAI key set:   ", bool(os.getenv("OPENAI_API_KEY")))
print("Anthropic key set:", bool(os.getenv("ANTHROPIC_API_KEY")))

## The universal wrapper

This is probably the single most useful function in LangChain:

```python
from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-4o-mini")
```

Notice the format: `provider:model_name`. LangChain looks at the prefix, picks the right integration package under the hood, and hands you back an object that behaves the same way no matter which provider you chose. That is what people mean when they say LangChain *abstracts* the model layer.

### My take on why this matters

This is **not magic**. Internally `init_chat_model` is essentially a switch statement that returns `ChatOpenAI(...)`, `ChatAnthropic(...)`, `ChatGoogleGenerativeAI(...)`, etc. The win is that *your* code never has to import a provider class. So when you decide tomorrow to A/B test a Claude model against a GPT model, you change one string and nothing else in your application changes. That is huge for production code where the model choice is not a one-time decision.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4o-mini")
model

## Same code, different provider

If you have an Anthropic key, run the cell below. Notice the code is identical except for the model name string. Every method we are about to use (`invoke`, `stream`, `batch`) behaves the same way regardless of which provider sits behind it.

If you only have one provider, that is fine. Skip and continue.

In [ ]:
if os.getenv("ANTHROPIC_API_KEY"):
    anthropic_model = init_chat_model("anthropic:claude-haiku-4-5-20251001")
    print(anthropic_model.invoke("Say hello in one short sentence.").text)
else:
    print("No Anthropic key set, skipping. The OpenAI model is enough for the rest of the notebook.")

## `invoke`: the basic call

`invoke` is the simplest method. Hand it some input, get back a single `AIMessage` once the model has finished generating.

You can pass it three different shapes of input and they all work:

1. a plain string
2. a list of LangChain message objects (`SystemMessage`, `HumanMessage`, ...)
3. a list of OpenAI-style dicts: `[{"role": "user", "content": "..."}]`

The reason all three work is that LangChain normalizes them all to the second form before sending to the provider. Pick whichever feels best for the moment. For quick experiments use strings. For real conversations with system prompts, use message objects.

In [ ]:
response = model.invoke("Why is the sky blue? Answer in one sentence.")
response

## What is actually in an `AIMessage`?

The thing `invoke` returns is not a string. It is an `AIMessage`, and it carries a lot more than just the text. Get used to looking at every one of these fields. They will be useful when you debug, when you log, and when you build agents.

In [ ]:
print("text:        ", response.text)
print("id:          ", response.id)
print("usage:       ", response.usage_metadata)
print("response_md: ", response.response_metadata)
print("type:        ", type(response).__name__)

### What to notice

- `response.text` is the assistant's reply as a plain string. Most of the time this is what you want.
- `response.content` is the *raw* content. It is sometimes a string and sometimes a list of typed blocks (text, reasoning, image, tool_call). `text` is a convenience that flattens that down for you.
- `usage_metadata` is the cleanest way to track tokens. Read it after every call if you care about cost. Format is `{'input_tokens': ..., 'output_tokens': ..., 'total_tokens': ...}`.
- `response_metadata` is provider-specific (model name, finish reason, system fingerprint). Useful for debugging, dangerous to depend on across providers.

## The four message types

A conversation in LangChain is just a list of messages. There are four types you will see constantly:

| Type | Who wrote it | When you use it |
|---|---|---|
| `SystemMessage` | You, before the conversation | Set the model's role, tone, hard rules |
| `HumanMessage` | The user | Each user turn |
| `AIMessage` | The model | The model's replies, and *also* its tool call requests |
| `ToolMessage` | Your code | The result of a tool the model asked for (notebook 3) |

The model only ever produces `AIMessage`. The other three are things *you* construct.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

conversation = [
    SystemMessage("You are a terse assistant. Answer in at most 10 words."),
    HumanMessage("What is the capital of France?"),
    AIMessage("Paris."),
    HumanMessage("And of Japan?"),
]

reply = model.invoke(conversation)
print(reply.text)

### Insight: why fake the past `AIMessage`?

Notice we put an `AIMessage("Paris.")` into the history above, even though we never asked the model that question. That is normal. The chat model is *stateless*. Every call gets the entire conversation as input. If you want the model to behave as if a previous turn happened, you append a fake `AIMessage` to the list. Some people use this to seed style or persona by putting an example exchange in the history.

## The three input formats, side by side

This is worth seeing once in one place. All three of these calls produce equivalent results:

In [ ]:
# 1. Plain string (single-turn, no system prompt)
r1 = model.invoke("Say 'ok' and nothing else.")

# 2. List of LangChain message objects
r2 = model.invoke([
    SystemMessage("You only say 'ok'."),
    HumanMessage("Hi."),
])

# 3. List of dicts in OpenAI's role/content format
r3 = model.invoke([
    {"role": "system", "content": "You only say 'ok'."},
    {"role": "user", "content": "Hi."},
])

print(r1.text, "|", r2.text, "|", r3.text)

## `stream`: get tokens as they generate

`invoke` waits until the entire response is done before returning. For chat UIs that feels slow. `stream` returns an iterator of `AIMessageChunk` objects, one per chunk the provider sends. Print them as they arrive and you get the typewriter effect users expect.

In [ ]:
for chunk in model.stream("Tell me a two-sentence story about a fox."):
    print(chunk.text, end="", flush=True)
print()

## Accumulating chunks back into a full message

Sometimes you want to stream to the screen *and* keep the final assembled message (for logging, history, token counts, etc). `AIMessageChunk` objects support `+`, so you can fold them as they come in:

In [ ]:
full = None
for chunk in model.stream("List three primary colors, comma separated."):
    full = chunk if full is None else full + chunk

print("final text:    ", full.text)
print("final usage:   ", full.usage_metadata)
print("type:          ", type(full).__name__)

**Insight**: `AIMessageChunk + AIMessageChunk` is associative, which is why the fold above works. Once you have the accumulated chunk, it has all the same fields a plain `AIMessage` would. Notice that `usage_metadata` is only populated on the *final* chunk, so accumulating is the only way to get token counts out of a stream.

## `batch`: many independent inputs at once

If you have a list of *independent* prompts, do not loop over `invoke`. Use `batch`. LangChain runs them in parallel up to a concurrency limit and returns the results in the same order as the inputs.

In [ ]:
prompts = [
    "Say the word 'red' and nothing else.",
    "Say the word 'green' and nothing else.",
    "Say the word 'blue' and nothing else.",
]

results = model.batch(prompts, config={"max_concurrency": 3})

for r in results:
    print(r.text)

## `batch_as_completed`: results as they finish

Same idea, but instead of waiting for the whole batch to be done, you get each result the moment it lands. The order is *not* preserved, so each yielded item is `(original_index, response)`.

Use this when latencies are uneven and you want to start showing partial results, e.g. a dashboard that summarizes 50 documents and updates as each finishes.

In [ ]:
for idx, r in model.batch_as_completed(prompts):
    print(f"index {idx}: {r.text}")

## Async versions

Every method has an `a`-prefixed async twin: `ainvoke`, `astream`, `abatch`. They take the exact same arguments. If you are building anything web-facing (FastAPI, websockets, an async agent loop), use these so a slow model call does not block your event loop.

In [ ]:
async def demo():
    r = await model.ainvoke("Say 'async works' and nothing else.")
    print(r.text)

await demo()

Note: in Jupyter, top-level `await` works because the notebook itself runs inside an event loop. In a regular `.py` file you would wrap it: `asyncio.run(demo())`.

## What can this model actually do? `model.profile`

Different providers and different models have different capabilities. Some support tool calling, some don't. Some take images, some don't. Rather than memorizing the matrix, ask the model itself:

In [ ]:
import json
print(json.dumps(model.profile, indent=2, default=str))

**Insight**: this matters more than it looks. If you write code that calls `bind_tools` on a model whose `profile` says `tool_calling: False`, you have a bug. In a serious app you would assert on `model.profile` before wiring up features so a wrong model swap fails loudly at startup instead of producing junk output later.

## Recap

You now have all the primitives we will use for the rest of the course:

- **One wrapper** (`init_chat_model`) for any provider
- **Four message types** that represent any conversation
- **Three call methods** (`invoke`, `stream`, `batch`) plus their async twins
- **One inspection trick** (`model.profile`) so you know what a given model supports

Next up in `02_prompts_and_chains.ipynb`: instead of building message lists by hand every time, we use `ChatPromptTemplate` to make them reusable and parameterized, and we wire them into models with the LCEL pipe operator. That is the move that lets you write `prompt | model | parser` and get a real callable chain.